In [ ]:
import joblib 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# pick which model to explain — comment out the others
TAG = "augmented"
# TAG = "qwen"; # TAG = "llama"
# TAG = "qwen_rewrite"; # TAG = "llama_rewrite"

b = joblib.load(f"xgb_bundle_{TAG}.joblib")
clf, tfidf         = b["clf"], b["tfidf"]
feat_names, hand_names = b["feat_names"], b["hand_names"]
Xte, te_x, te_y    = b["Xte"], b["te_x"], np.array(b["te_y"])
hte, htr, tr_y     = b["hte"], b["htr"], np.array(b["tr_y"])
prob, pred         = np.array(b["prob"]), np.array(b["pred"])
print(f"Loaded {TAG}: {Xte.shape[0]} test notes, {len(feat_names)} features")

# Q1: Which features did the generator reproduce worst? (SHAP)

In [ ]:
import shap
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

In [ ]:
# --- global SHAP (prediction-based) ---
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(Xte)
mean_abs = np.abs(shap_values).mean(axis=0)

print(f"=== {TAG}: Top 20 features by mean |SHAP| (label 1 = synthetic) ===")
for i in np.argsort(mean_abs)[::-1][:20]:
    print(f"  {feat_names[i]:28s} {mean_abs[i]:.4f}")

# --- global PFI (loss-based) on the hand-crafted block only ---
# (PFI over 500 sparse tf-idf dims is noisy; run it on interpretable features)
clf_hand = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.5, reg_alpha=1.0, reg_lambda=2.0,
    eval_metric="logloss", random_state=42).fit(htr.values, tr_y)
pfi = permutation_importance(clf_hand, hte.values, te_y, scoring="roc_auc",
                             n_repeats=10, random_state=42)
print(f"\n=== {TAG}: PFI on hand-crafted features (AUC drop when permuted) ===")
for i in np.argsort(pfi.importances_mean)[::-1]:
    print(f"  {hand_names[i]:28s} {pfi.importances_mean[i]:+.4f} ± {pfi.importances_std[i]:.4f}")

# --- SHAP summary plot (beeswarm: magnitude + direction) ---
shap.summary_plot(shap_values, Xte, feature_names=feat_names, max_display=20, show=False)
plt.title(f"SHAP summary — {TAG}"); plt.tight_layout()
plt.savefig(f"q1_shap_{TAG}.png", dpi=150, bbox_inches="tight"); plt.show()